# Ingestion Pipeline

End-to-end pipeline that:
1. Loads `.txt` knowledge-base files
2. Parses metadata from file headers
3. Chunks the documents
4. Embeds the chunks
5. Stores them in the vector database
6. Runs a sanity-check retrieval query

**Reuses** logic from `1_load_chunk.ipynb`, `2_embeddings.ipynb`, `3_retrieval.ipynb`, `4_basic_RAG.ipynb` and `src/` py files.

## Setup 
### Imports & Paths

In [ ]:
# !pip install -r ../requirements.txt


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
from pathlib import Path
import sys
import os
import numpy as np
from dotenv import load_dotenv

# ── Project root & src on path ──────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# ── Knowledge-base directories ───────────────────────────────────────────────
LGBT_EU_BY_COUNTRY_DIR = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_country"
LGBT_EU_BY_SUBSET_DIR  = PROJECT_ROOT / "data" / "3_txt_KB" / "LGBT_EU" / "by_subset"
HIV_KB_DIR             = PROJECT_ROOT / "data" / "3_txt_KB" / "HIV_AIDS_data"
UNICEF_KB_DIR          = PROJECT_ROOT / "data" / "3_txt_KB" / "UNICEF_Immunization"

KB_DIRS = [
    LGBT_EU_BY_COUNTRY_DIR,
    LGBT_EU_BY_SUBSET_DIR,
    HIV_KB_DIR,
    UNICEF_KB_DIR,
]

print("Project root:", PROJECT_ROOT)
for d in KB_DIRS:
    status = "✓" if d.exists() else "✗ (not found)"
    print(f"  {status}  {d.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG
  ✓  data\3_txt_KB\LGBT_EU\by_country
  ✓  data\3_txt_KB\LGBT_EU\by_subset
  ✓  data\3_txt_KB\HIV_AIDS_data
  ✓  data\3_txt_KB\UNICEF_Immunization


In [61]:
# ── Standard library & third-party ───────────────────────────────────────────
from typing import Dict, List, Tuple

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# ── src modules (reused from previous notebooks) ─────────────────────────────
from vectorstore import build_vectorstore
from retrieval   import retrieve, print_results, format_context
from llm         import build_prompt, ask


In [62]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", "✅" if OPENAI_API_KEY else "❌ NOT FOUND")

API key loaded: ✅


In [63]:
# used in evaluation stage
import nltk
import re
import numpy as np
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from textstat import flesch_reading_ease

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pathlib import Path

## Parameters

- uses `RecursiveCharacterTextSplitter` used in `1_load_chunk.ipynb`.
- Edit params to change how it will be applied.

In [64]:
# whether we actually use OpenAI credits on this
USE_OPENAI = True

# ── Chunking parameters (mirrors 1_load_chunk.ipynb) ─────────────────────────
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# ── Vector-store persistence path ─────────────────────────────────────────────
VECTORSTORE_DIR = PROJECT_ROOT / "data" / "vectorstore"

# ── Embedding model (mirrors 2_embeddings.ipynb) ──────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-small"

# ── Retrieval parameters (mirrors 3_retrieval.ipynb) ─────────────────────────
TOP_K = 18

print(f"CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}")
print(f"EMBEDDING_MODEL={EMBEDDING_MODEL}")
print(f"VECTORSTORE_DIR={VECTORSTORE_DIR}")

CHUNK_SIZE=500, CHUNK_OVERLAP=50
EMBEDDING_MODEL=text-embedding-3-small
VECTORSTORE_DIR=C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


## Load All `.txt` Files
- load in from knowledge base (the output of `7_generate_text_knowledge_base.ipynb`)

In [65]:
def collect_txt_files(directories: List[Path]) -> List[Path]:
    """Recursively collect every .txt file from the given directories."""
    files: List[Path] = []
    for directory in directories:
        if not directory.exists():
            print(f"  ⚠  Directory not found, skipping: {directory}")
            continue
        found = sorted(directory.rglob("*.txt"))
        print(f"  Found {len(found):>4} files in {directory.relative_to(PROJECT_ROOT)}")
        files.extend(found)
    return files


all_txt_files = collect_txt_files(KB_DIRS)
print(f"\nTotal .txt files: {len(all_txt_files)}")

  Found 4427 files in data\3_txt_KB\LGBT_EU\by_country
  Found  699 files in data\3_txt_KB\LGBT_EU\by_subset
  Found   69 files in data\3_txt_KB\HIV_AIDS_data
  Found  292 files in data\3_txt_KB\UNICEF_Immunization

Total .txt files: 5487


## Handle Metadata + Content

In [66]:
def parse_document(file_path: Path) -> Tuple[str, Dict[str, str]]:
    raw = file_path.read_text(encoding="utf-8")
    lines = raw.splitlines()

    metadata: Dict[str, str] = {}
    content_start = 0

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "":          # blank line → header ends here
            content_start = i + 1
            break

        if ":" in stripped:         # metadata line  KEY: value
            key, _, value = stripped.partition(":")
            metadata[key.strip().lower()] = value.strip()
        else:
            # Not a metadata line and not blank → no header, treat whole file as content
            content_start = 0
            metadata = {}
            break

    content = "\n".join(lines[content_start:]).strip()
    metadata["Source"] = str(file_path)

    return content, metadata

In [67]:
# ── Quick smoke-test on the first available file ──────────────────────────────
if all_txt_files:
    _sample_content, _sample_meta = parse_document(all_txt_files[0])
    print("Sample file :", all_txt_files[0].name)
    print("Metadata    :", _sample_meta)
    print("Content (100 chars):", _sample_content[:100], "...")
else:
    print("No files found — check KB_DIRS above.")

Sample file : b1_a_answer_by_Austria.txt
Metadata    : {'dataset': 'EU_LGBT', 'question_code': 'b1_a', 'subset': 'Austria', 'Source': 'C:\\Users\\RAZER\\Desktop\\portfolio-projects\\1. RAG\\data\\3_txt_KB\\LGBT_EU\\by_country\\LGBT_Survey_DailyLife\\b1_a_answer_by_Austria.txt'}
Content (100 chars): Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual a ...


In [68]:
def build_documents(file_paths: List[Path]) -> List[Document]:
    """
    Parse every file and return a list of LangChain Documents.
    Mirrors the Document creation pattern from 1_load_chunk.ipynb.
    """
    docs: List[Document] = []
    errors: List[str] = []

    for fp in file_paths:
        try:
            content, metadata = parse_document(fp)
            if content:            # skip empty files
                docs.append(Document(page_content=content, metadata=metadata))
        except Exception as exc:
            errors.append(f"{fp.name}: {exc}")

    if errors:
        print(f"⚠  {len(errors)} file(s) could not be parsed:")
        for e in errors:
            print("  ", e)

    print(f"\nDocuments created : {len(docs)}")
    return docs


raw_documents = build_documents(all_txt_files)


Documents created : 5487


## Chunk Documents

Reuses the `RecursiveCharacterTextSplitter` we made `1_load_chunk.ipynb`.

In [69]:
# ── Text splitter — mirrors 1_load_chunk.ipynb ────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True,   # keeps track of character offset (used in 1_load_chunk.ipynb)
)

chunks = text_splitter.split_documents(raw_documents)

print(f"Raw documents : {len(raw_documents)}")
print(f"Chunks        : {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // max(len(chunks), 1)} chars")

Raw documents : 5487
Chunks        : 30830
Avg chunk size: 352 chars


In [70]:
# ── Inspect a sample chunk ────────────────────────────────────────────────────
if chunks:
    sample = chunks[0]
    print("Sample chunk metadata :", sample.metadata)
    print("Sample chunk content  :", sample.page_content[:200], "...")

Sample chunk metadata : {'dataset': 'EU_LGBT', 'question_code': 'b1_a', 'subset': 'Austria', 'Source': 'C:\\Users\\RAZER\\Desktop\\portfolio-projects\\1. RAG\\data\\3_txt_KB\\LGBT_EU\\by_country\\LGBT_Survey_DailyLife\\b1_a_answer_by_Austria.txt', 'start_index': 0}
Sample chunk content  : Question b1_a — In your opinion, how widespread is offensive language about lesbian, gay, bisexual and/or transgender people by politicians in the country where you live? | Austria responses (Bisexual ...


## Embed Documents
- Reuses the `OpenAIEmbeddings` setup from `2_embeddings.ipynb`.

In [71]:
# ── Embedding model — mirrors 2_embeddings.ipynb ──────────────────────────────
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Quick sanity check: embed a single string
_test_vec = embeddings.embed_query("test")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Vector dimension: {len(_test_vec)}")

Embedding model : text-embedding-3-small
Vector dimension: 1536


## Store in Vector DB
- Reuses `load_vectorstore` from `src/vectorstore.py`.

In [72]:
# ── Persist directory ─────────────────────────────────────────────────────────
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Vector-store directory: {VECTORSTORE_DIR}")

Vector-store directory: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


In [73]:
# ── Build / overwrite the vector store ───────────────────────────────────────
# load_vectorstore is expected to accept (chunks, embeddings, persist_directory)
# and return a Chroma (or equivalent) vectorstore — as used in 3_retrieval.ipynb

vectorstore = build_vectorstore(
    documents=chunks,
    embeddings=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)

print(f"\n✓ Vector store built and persisted to: {VECTORSTORE_DIR}")
print(f"  Total vectors stored: {vectorstore._collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Built vectorstore: 123320 chunks

✓ Vector store built and persisted to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore
  Total vectors stored: 123320


# Testing
## Small test of End-to-End Retrieval
- Runs a sample query through the full pipeline
- Same structure as `4_basic_RAG.ipynb`.

In [74]:
# SAMPLE_QUERY = "What is the HIV prevalence rate in Eastern Europe?"
# SAMPLE_QUERY = "Do Lesbians in Romania experience a better or worse daily life experience than Bisexual women in Romania?"
SAMPLE_QUERY = "Lesbian vs Bisexual women, daily life experience in Romania"
print(f"Sample query: {SAMPLE_QUERY}")

Sample query: Lesbian vs Bisexual women, daily life experience in Romania


### 1. Retrieve relevant chunks


In [75]:
# retrieve() mirrors 3_retrieval.ipynb usage
results = retrieve(
    query=SAMPLE_QUERY,
    vectorstore=vectorstore,
    k=TOP_K,
)

print(f"Retrieved {len(results)} chunks:\n")
print_results(query = SAMPLE_QUERY, results = results)

Retrieved 18 chunks:

Query: 'Lesbian vs Bisexual women, daily life experience in Romania'

--- Result 1 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\b2_a_answer_by_Bisexualwomen.txt
Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disagree: 0%, Current situation is fine: 1%, Don`t know: 7%

--- Result 2 ---
Source: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\LGBT_EU\by_subset\LGBT_Survey_DailyLife\b2_a_answer_by_Bisexualwomen.txt
Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | 

### 2. Format context

In [76]:
context = format_context(results)
print("Context passed to LLM (first 500 chars):")
print(context[:500], "...")

Context passed to LLM (first 500 chars):
Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disagree: 0%, Current situation is fine: 1%, Don`t know: 7%

Question b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you l ...


### 3. Build prompt

In [77]:
# build_prompt() mirrors 4_basic_RAG.ipynb usage
prompt = build_prompt(query=SAMPLE_QUERY, context=context)
print("Prompt (first 500 chars):")
print(prompt[:500], "...")

Prompt (first 500 chars):
[SystemMessage(content="You are a helpful assistant. Answer the user's question using only the context provided below. If the answer is not in the context, say 'I don't have enough information to answer that.'", additional_kwargs={}, response_metadata={}), HumanMessage(content='Context:\nQuestion b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disagree: 0%, Current situation is fine: 1%, Don`t know: 7%\n\nQuestion b2_a — What would allow you to be more comfortable living as a lesbian, gay or bisexual person in the country where you live? Anti-discrimination policies referring to sexual orientation at the workplace? | Bisexual women responses in Romania: Strongly agree: 61%, Agree: 27%, Disagree: 3%, Strongly disa

### 4. Ask the LLM

In [78]:
# ask() mirrors 4_basic_RAG.ipynb usage
answer = ask(prompt, context)
print("=" * 60)
print("QUESTION:", SAMPLE_QUERY)
print("=" * 60)
print("ANSWER:")
print(answer)

QUESTION: Lesbian vs Bisexual women, daily life experience in Romania
ANSWER:
I don't have enough information to answer that.


## Pipeline Summary

In [79]:
print("Pipeline complete ✓")
print(f"  Files loaded    : {len(all_txt_files)}")
print(f"  Documents parsed: {len(raw_documents)}")
print(f"  Chunks created  : {len(chunks)}")
print(f"  Vectors stored  : {vectorstore._collection.count()}")
print(f"  Vector store at : {VECTORSTORE_DIR}")

Pipeline complete ✓
  Files loaded    : 5487
  Documents parsed: 5487
  Chunks created  : 30830
  Vectors stored  : 123320
  Vector store at : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\vectorstore


## RAG Self-Evaluation

In [80]:
# Install lightweight evaluation dependencies (skip if already present)
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("textstat")
_ensure("openpyxl")
_ensure("nltk")

nltk.download("punkt",      quiet=True)
nltk.download("punkt_tab",  quiet=True)

True

In [81]:
TEST_QUESTIONS = [
    # LGBT EU survey
    "What percentage of gay men in Germany experienced discrimination in the past year?",
    "How comfortable do lesbian women in France feel being open about their identity at work?",
    "What share of transgender people in Poland reported hate-motivated violence?",
    "Compare acceptance levels of same-sex couples in Sweden versus Hungary.",
    # HIV / AIDS
    "What is the HIV prevalence rate among adults in sub-Saharan Africa?",
    "How has antiretroviral therapy coverage changed over the past decade?",
    # UNICEF Immunization
    "What is the global vaccination coverage rate for measles in children under five?",
    "Which regions have the lowest DTP3 immunization rates according to UNICEF data?",
]
print(f"Running evaluation on {len(TEST_QUESTIONS)} questions...")

Running evaluation on 8 questions...


In [82]:
def _tokenize(text: str) -> set:
    """Lowercase word tokens, punctuation stripped."""
    return set(re.findall(r"\b[a-z]+\b", text.lower()))


def retrieval_metrics(query: str, chunks, embeddings_model) -> dict:
    """Compute retrieval-level metrics for a list of LangChain Documents."""
    texts = [c.page_content for c in chunks]

    # Duplicate ratio
    unique_texts = set(texts)
    dup_ratio = round(1 - len(unique_texts) / len(texts), 3) if texts else 0.0

    # Unique source files
    sources = [c.metadata.get("Source", "") for c in chunks]
    unique_src = len(set(sources))

    # Cosine similarity between query embedding and chunk embeddings
    try:
        q_vec  = np.array(embeddings_model.embed_query(query)).reshape(1, -1)
        c_vecs = np.array(embeddings_model.embed_documents(list(unique_texts)))
        cos_scores = cosine_similarity(q_vec, c_vecs)[0]
        cos_avg = round(float(cos_scores.mean()), 4)
    except Exception:
        cos_avg = None

    return {
        "# Chunks Retrieved": len(texts),
        "Unique Sources":     unique_src,
        "Duplicate Ratio":    dup_ratio,
        "Query Chunk Cosine Avg": cos_avg,
    }


def answer_metrics(query: str, answer: str, chunks) -> dict:
    """Compute answer-quality metrics."""
    no_info_phrases = [
        "don't have enough information",
        "do not have enough information",
        "cannot answer",
        "not in the context",
    ]
    answered = not any(p in answer.lower() for p in no_info_phrases)

    answer_words = _tokenize(answer)
    chunk_words  = set()
    for c in chunks:
        chunk_words |= _tokenize(c.page_content)
    query_words = _tokenize(query)

    groundedness   = round(len(answer_words & chunk_words) / len(answer_words), 3) if answer_words else 0.0
    relevancy      = round(len(answer_words & query_words) / len(query_words),  3) if query_words  else 0.0
    word_count     = len(answer.split())
    try:
        flesch = round(flesch_reading_ease(answer), 1)
    except Exception:
        flesch = None

    return {
        "Answer Length Words":  word_count,
        "Answered":             answered,
        "Groundedness":         groundedness,
        "Relevancy Score":      relevancy,
        "Flesch Reading Ease":  flesch,
    }

In [ ]:
# Formatting for the Excel Output, will be used later
HEADER_FILL  = PatternFill("solid", start_color="698237", end_color="698237")   # dark green 5a7540
ALT_FILL     = PatternFill("solid", start_color="EAFACA", end_color="EAFACA")   # light green
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=11)
BODY_FONT    = Font(name="Arial", size=10)
WRAP_ALIGN   = Alignment(wrap_text=True, vertical="top")
CENTER_ALIGN = Alignment(horizontal="center", vertical="top")
THIN_BORDER  = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin"),
)

# Colour coding for 'answered' column on Summary sheet
GREEN_FILL = PatternFill("solid", start_color="C6EFCE", end_color="C6EFCE")
RED_FILL   = PatternFill("solid", start_color="FFC7CE", end_color="FFC7CE")

In [89]:
# Re-use the same embedding model that was used to build the vectorstore
# embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

summary_rows   = []   # one dict per question  → Sheet 1
retrieval_rows = []   # one dict per chunk     → Sheet 2

for q_idx, question in enumerate(TEST_QUESTIONS, start=1):
    print(f"[{q_idx}/{len(TEST_QUESTIONS)}] {question[:80]}...")

    # 1. Retrieve
    chunks   = retrieve(query=question, vectorstore=vectorstore, k=TOP_K)
    ctx      = format_context(chunks)

    # 2. Generate answer
    prompt   = build_prompt(query=question, context=ctx)
    answer   = ask(prompt, ctx)

    # 3. Compute metrics
    r_metrics = retrieval_metrics(question, chunks, embeddings)
    a_metrics = answer_metrics(question, answer, chunks)

    # 4. Accumulate summary row
    row = {"Q ID": q_idx, "Question": question, "answer": answer}
    row.update(r_metrics)
    row.update(a_metrics)
    summary_rows.append(row)

    # 5. Accumulate chunk-level rows
    seen_texts = set()
    for rank, chunk in enumerate(chunks, start=1):
        text = chunk.page_content
        is_dup = text in seen_texts
        seen_texts.add(text)
        retrieval_rows.append({
            "Q ID":      q_idx,
            "Question":  question,
            "Rank":      rank,
            "Source":    chunk.metadata.get("Source", ""),
            "Is Duplicate": is_dup,
            "Chunk Text": text,
        })

[1/8] What percentage of gay men in Germany experienced discrimination in the past yea...
[2/8] How comfortable do lesbian women in France feel being open about their identity ...
[3/8] What share of transgender people in Poland reported hate-motivated violence?...
[4/8] Compare acceptance levels of same-sex couples in Sweden versus Hungary....
[5/8] What is the HIV prevalence rate among adults in sub-Saharan Africa?...
[6/8] How has antiretroviral therapy coverage changed over the past decade?...
[7/8] What is the global vaccination coverage rate for measles in children under five?...
[8/8] Which regions have the lowest DTP3 immunization rates according to UNICEF data?...


In [90]:
OUTPUT_PATH_EVAL = PROJECT_ROOT / "RAG_evaluation.xlsx"

# ── Dataframes ───────────────────────────────────────────────────────────────
df_summary   = pd.DataFrame(summary_rows)
df_retrieval = pd.DataFrame(retrieval_rows)

SCORE_EXPLANATIONS = [
    {"Metric": "# Chunks Retrieved",       "sheet": "Summary / Retrieval_Detail", "Description": "Total number of chunks returned by the vector store for this query. Controlled by TOP_K."},
    {"Metric": "Unique Sources",           "sheet": "Summary",                    "Description": "Number of distinct source files among the retrieved chunks. Higher = more diverse retrieval."},
    {"Metric": "Duplicate Ratio",          "sheet": "Summary",                    "Description": "Fraction of retrieved chunks that are exact-text duplicates. 0 = no duplicates, 1 = all duplicates. Lower is better."},
    {"Metric": "Query Chunk Cosine Avg",   "sheet": "Summary",                    "Description": "Mean cosine similarity (0–1) between the query embedding and each unique chunk embedding. Higher means chunks are semantically closer to the query."},
    {"Metric": "Answer Length Words",      "sheet": "Summary",                    "Description": "Word count of the generated answer. Very short answers may indicate the model couldn't answer."},
    {"Metric": "Answered",                 "sheet": "Summary",                    "Description": "TRUE if the LLM returned a substantive answer; FALSE if it said it lacks enough information."},
    {"Metric": "Groundedness",             "sheet": "Summary",                    "Description": "Fraction of words in the answer that also appear in the retrieved chunks. Higher = answer is more grounded in retrieved evidence."},
    {"Metric": "Relevancy Score",          "sheet": "Summary",                    "Description": "Fraction of query words that appear in the answer. Higher = answer directly addresses the question."},
    {"Metric": "Flesch Reading Ease",      "sheet": "Summary",                    "Description": "Flesch Reading Ease score. 60–70 = standard; higher = easier to read; lower = more complex text."},
    {"Metric": "Is Duplicate (Retrieval)", "sheet": "Retrieval_Detail",           "Description": "TRUE if this chunk's text was already seen at a higher rank for the same question."},
]
df_scores = pd.DataFrame(SCORE_EXPLANATIONS)

# ── Write raw data via pandas ExcelWriter ────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH_EVAL, engine="openpyxl") as writer:
    df_summary.to_excel(writer,   sheet_name="Summary",           index=False)
    df_retrieval.to_excel(writer, sheet_name="Retrieval_Detail",  index=False)
    df_scores.to_excel(writer,    sheet_name="Score_Explanation", index=False)

# ── Apply formatting with openpyxl ───────────────────────────────────────────
wb = load_workbook(OUTPUT_PATH_EVAL)

In [91]:
def _style_sheet(ws, col_widths: dict, wrap_cols: list = None):
    wrap_cols = wrap_cols or []
    for row_idx, row in enumerate(ws.iter_rows(), start=1):
        for cell in row:
            cell.font   = HEADER_FONT if row_idx == 1 else BODY_FONT
            cell.border = THIN_BORDER
            if row_idx == 1:
                cell.fill      = HEADER_FILL
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            else:
                if cell.column_letter in wrap_cols:
                    cell.alignment = WRAP_ALIGN
                else:
                    cell.alignment = Alignment(vertical="top")
                if row_idx % 2 == 0:
                    cell.fill = ALT_FILL
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width
    ws.freeze_panes = "A2"


# -- Summary sheet --
ws_sum = wb["Summary"]
ws_sum.row_dimensions[1].height = 30
_style_sheet(ws_sum,
    col_widths={"A": 6, "B": 55, "C": 70, "D": 18, "E": 16,
                "F": 16, "G": 22, "H": 18, "I": 12, "J": 18, "K": 18, "L": 20},
    wrap_cols=["B", "C"]
)
# Colour-code 'answered' column (col I = index 9, 1-based)
answered_col = [c.column for c in ws_sum[1] if c.value == "answered"]
if answered_col:
    col_letter = get_column_letter(answered_col[0])
    for row in ws_sum.iter_rows(min_row=2, min_col=answered_col[0], max_col=answered_col[0]):
        for cell in row:
            if cell.value is True:
                cell.fill = GREEN_FILL
            elif cell.value is False:
                cell.fill = RED_FILL

# -- Retrieval Detail sheet --
ws_ret = wb["Retrieval_Detail"]
_style_sheet(ws_ret,
    col_widths={"A": 6, "B": 55, "C": 6, "D": 60, "E": 13, "F": 80},
    wrap_cols=["B", "D", "F"]
)

# -- Score Explanation sheet --
ws_exp = wb["Score_Explanation"]
_style_sheet(ws_exp,
    col_widths={"A": 28, "B": 30, "C": 90},
    wrap_cols=["C"]
)
for row in ws_exp.iter_rows(min_row=2):
    ws_exp.row_dimensions[row[0].row].height = 40

wb.save(OUTPUT_PATH_EVAL)
print(f"   Saved: {OUTPUT_PATH_EVAL}")
print(f"   Sheets: Summary ({len(summary_rows)} rows)  "
      f"| Retrieval_Detail ({len(retrieval_rows)} rows)  "
      f"| Score_Explanation ({len(SCORE_EXPLANATIONS)} rows)")

   Saved: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\RAG_evaluation.xlsx
   Sheets: Summary (8 rows)  | Retrieval_Detail (144 rows)  | Score_Explanation (10 rows)


In [92]:
display_cols = [
    "Q ID", "Question", "Unique Sources", "Duplicate Ratio", "Query Chunk Cosine Avg","Answered", "Groundedness", "Relevancy Score", "Flesch Reading Ease",
]

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.3f}".format)

df_display = df_summary[display_cols].copy()
df_display["Question"] = df_display["Question"].str[:55] + "..."

df_display.head(5)

,Q ID,Question,Unique Sources,Duplicate Ratio,Query Chunk Cosine Avg,Answered,Groundedness,Relevancy Score,Flesch Reading Ease
0,1,What percentage of gay men in Germany experienced discr...,3,0.722,0.727,False,0.444,0.000,61.200
1,2,How comfortable do lesbian women in France feel being o...,4,0.722,0.673,False,0.222,0.000,61.200
2,3,What share of transgender people in Poland reported hat...,5,0.722,0.694,False,0.222,0.000,61.200
3,4,Compare acceptance levels of same-sex couples in Sweden...,5,0.722,0.659,False,0.333,0.000,61.200
4,5,What is the HIV prevalence rate among adults in sub-Sah...,6,0.778,0.359,False,0.000,0.000,61.200
